In [1]:
#this notebook is used to create the baseline calculations for each "followed" token (configured in the config file), using spark
#setup installing package to the venv
%pip install pandas pyarrow pyspark
%pip install install-jdk

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
#java home setup
#this is importent for the venv, not part of our project
%pip install install-jdk

import os
import sys
import glob
import jdk

# 1. Target the .venv directory of the current running Python kernel
venv_path = sys.prefix
jvm_dir = os.path.join(venv_path, "jvm")

# 2. Download/ensure JDK 17 exists for THIS current OS/architecture
jdk.install('17', path=jvm_dir)

# 3. Dynamically search for the 'bin/java' executable regardless of OS directory nesting
java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java"), recursive=True)
if not java_execs:
    # Check for Windows .exe just in case
    java_execs = glob.glob(os.path.join(jvm_dir, "**/bin/java.exe"), recursive=True)

if not java_execs:
    raise RuntimeError(f"JDK binary could not be found inside {jvm_dir}")

# The true JAVA_HOME is the parent directory of 'bin'
resolved_java_home = os.path.dirname(os.path.dirname(os.path.abspath(java_execs[0])))

# 4. Set environment variables for the active session
os.environ["JAVA_HOME"] = resolved_java_home
os.environ["PATH"] = os.path.join(resolved_java_home, "bin") + os.pathsep + os.environ.get("PATH", "")

print(f"Universal JDK configured at: {resolved_java_home}")

Note: you may need to restart the kernel to use updated packages.


JdkError: [Errno 13] Permission denied: '/home/wnder/Documents/repos/teleSpikeRedo/.venv/jvm/jdk-17.0.20.1+1/lib/server/classes_nocoops.jsa'

In [ ]:
import re
import json
import sqlite3
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as SqlFun
from pyspark.sql.types import ArrayType, StringType

with open("config.json", "r") as f:
    config = json.load(f)
#def GetBaseline(token):

# a simple worker functaion for parsing one message into tokes
def tokenize(text):
    if not text:
        return []
    #Hebrew and alphanumeric words of length >= 2
    return re.findall(r"[\u0590-\u05fe\w]{2,}", text.lower())

#conver time stamp into hour and date
def add_time_columns(df):
    return df.withColumn("dt", SqlFun.to_timestamp(SqlFun.col("ts"))) \
             .withColumn("hour", SqlFun.hour(SqlFun.col("dt"))) \
             .withColumn("date", SqlFun.to_date(SqlFun.col("dt")))

#tokenise and exploead 
def explode_and_filter_tokens(df, allowed_tokens=None):
    tokenize_udf = SqlFun.udf(tokenize, ArrayType(StringType()))
    
    tokens_df = df.withColumn("word", SqlFun.explode(tokenize_udf(SqlFun.col("text"))))
    
    if allowed_tokens:
        tokens_df = tokens_df.filter(SqlFun.col("word").isin(allowed_tokens))
        
    return tokens_df

def compute_hourly_pivots(tokens_df, total_days):
    # group rows per word and hour
    counts = tokens_df.groupBy("word", "hour").count()
    
    # Pivot hours into columns (now each word has a clolumb for each time of day)
    pivoted = counts.groupBy("word").pivot("hour", list(range(24))).sum("count").na.fill(0)
    
    # divide counts by total days to get average baseline per hour
    # Rename columns to h0, h1 ... h23
    for h in range(24):
        pivoted = pivoted.withColumn(f"h{h}", SqlFun.col(str(h)) / total_days).drop(str(h))
        
    return pivoted

#main function calling other parts    
def generate_baseline_table(sqlite_path, output_table_path, allowed_tokens):
    #this will recive an sql full of thounsds or millions of messages and a 
    #list of importent tokens. it will clean and tokenise each message, filter not importent tokens like "and" "if". any thing that isnt in the list.
    #it will then save in a small sql table
    #a row for each token with a columb for each hour of the day, and save the avrage apperenses in that hour for each token. we can than devide by 60 or 240 to get baselines for our time window

    spark = SparkSession.builder \
        .appName("BaselineGenerator") \
        .config("spark.driver.memory", "2g") \
        .getOrCreate()

    # load messages from local data base into Spark
    conn = sqlite3.connect(sqlite_path)
    pdf = pd.read_sql_query("SELECT text, ts FROM messages", conn)
    conn.close()

    if pdf.empty:
        print("No messages found.")
        return

    raw_df = spark.createDataFrame(pdf)

    #add time of day columb to data frame, parsed from the timestamp that came with the message
    #as well as a date columb for later calcualtions
    timed_df = add_time_columns(raw_df)

    # Calculate total days for avrge calculations later on
    total_days = max(1, timed_df.select("date").distinct().count())

    # tokenization and explode into rows
    tokens_df = explode_and_filter_tokens(timed_df, allowed_tokens=allowed_tokens)

    # use the data to turn the table into each row has: word h0 h1...., in each cloumb have the avrage for that time of day
    baseline_matrix = compute_hourly_pivots(tokens_df, total_days)

    # save to sql
    baseline_pdf = baseline_matrix.toPandas()
    
    conn_out = sqlite3.connect(output_table_path)
    baseline_pdf.to_sql("token_baselines", conn_out, if_exists="replace", index=False)
    conn_out.close()
    
    print(f"Generated baselines for {len(baseline_pdf)} tokens over {total_days} days.")
    return baseline_pdf

In [ ]:
#create baselines.db
conn = sqlite3.connect(config["baselines_db_path"])
cursor = conn.cursor()

hour_cols = ", ".join([f"h{i} REAL" for i in range(24)])

cursor.execute(f"""
CREATE TABLE IF NOT EXISTS token_baselines (
    word TEXT,
    {hour_cols}
)
""")

conn.commit()
conn.close()

print("Empty token_baselines table created.")

In [ ]:

# Run the pipeline on your scraped messages
baselines_df = generate_baseline_table(
    sqlite_path=config["messages_db_path"],#"messages.db",
    output_table_path=config["baselines_db_path"],
    allowed_tokens=config["followed_tokens"]  # Set to a list like ["טיל", "אזעקה"] if you want to filter, or None for all
)

# Preview the top tokens at 14:00 (2:00 PM)
if baselines_df is not None:
    print(baselines_df[["word", "h14"]].sort_values(by="h14", ascending=False).head(10))

In [ ]:
#print data base messages.db
conn = sqlite3.connect(config["messages_db_path"])
df_msgs = pd.read_sql_query("SELECT * FROM messages LIMIT 20", conn)
conn.close()
print("Messages Table:")
display(df_msgs)

In [ ]:
#print basline.db
conn = sqlite3.connect(config["baselines_db_path"])
#df_base = pd.read_sql_query("SELECT word, h0, h8, h14, h20 FROM token_baselines ORDER BY h14 DESC LIMIT 10", conn)
df_base = pd.read_sql_query("SELECT * FROM token_baselines ORDER BY h14 DESC LIMIT 10", conn)
conn.close()
print("Baselines Table:")
display(df_base)